<a href="https://colab.research.google.com/github/jealim0923-art/welcome/blob/main/%EC%8B%A4%EC%8A%B5%EA%B3%BC%EC%A0%9C_4_%EC%83%81_%EA%B9%80%EC%9E%AC%EB%A6%BC%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─────────────────────────────────────────
# 1. 데이터 로드 및 BTC 필터링
# ─────────────────────────────────────────
df = pd.read_csv("crypto_prices_sample.csv", parse_dates=["date"])
btc = df[df["symbol"] == "BTC"].sort_values("date").reset_index(drop=True)

print("=" * 55)
print("[ BTC 데이터 요약 ]")
print(f"  기간: {btc['date'].min().date()} ~ {btc['date'].max().date()}")
print(f"  총 거래일: {len(btc)}일")
print(f"  시작가: ${btc['close'].iloc[0]:,.0f}  |  종료가: ${btc['close'].iloc[-1]:,.0f}")
print("=" * 55)

[ BTC 데이터 요약 ]
  기간: 2025-11-01 ~ 2025-12-30
  총 거래일: 60일
  시작가: $52,000  |  종료가: $49,077


In [ ]:
# ─────────────────────────────────────────
# 2. MA7 예측값 생성
#    예측[i] = close[i-7 : i].mean()  → i번째 날의 예측
#    즉, "어제까지 7일 평균 = 오늘 예측값"
# ─────────────────────────────────────────
WINDOW = 7
btc["ma7"] = btc["close"].rolling(window=WINDOW).mean()

# shift(1): 예측은 전날 MA를 당일 예측으로 사용
btc["predicted"] = btc["ma7"].shift(1)

In [ ]:
# 사용 데이터: BTC 일별 종가 (2025-11-01 ~ 2025-12-30, 60일)
# 예측 방식: MA7 (7일 이동평균, 다음 날 종가 = 직전 7일 평균)
# 평가 지표: MAE(평균절대오차), RMSE, MAPE / 테스트 구간: 마지막 14일

import pandas as pd
import numpy as np

In [ ]:
# ─────────────────────────────────────────
# 3. 테스트 구간 설정 (마지막 14일)
# ─────────────────────────────────────────
TEST_DAYS = 14
test = btc.dropna(subset=["predicted"]).tail(TEST_DAYS).copy()

print("\n[ MA7 예측 vs 실제값 비교 (마지막 14일) ]")
print(f"{'날짜':<12} {'실제값':>12} {'예측값':>12} {'오차(실제-예측)':>16} {'오차율':>8}")
print("-" * 62)
for _, row in test.iterrows():
    err = row["close"] - row["predicted"]
    err_pct = abs(err) / row["close"] * 100
    print(f"{str(row['date'].date()):<12} ${row['close']:>11,.0f} ${row['predicted']:>11,.0f} {err:>+15,.0f}  {err_pct:>6.2f}%")



[ MA7 예측 vs 실제값 비교 (마지막 14일) ]
날짜                    실제값          예측값        오차(실제-예측)      오차율
--------------------------------------------------------------
2025-12-17   $     48,846 $     49,823            -977    2.00%
2025-12-18   $     48,616 $     49,720          -1,104    2.27%
2025-12-19   $     49,176 $     49,570            -394    0.80%
2025-12-20   $     49,364 $     49,442             -78    0.16%
2025-12-21   $     48,458 $     49,328            -870    1.80%
2025-12-22   $     48,636 $     49,091            -455    0.94%
2025-12-23   $     48,446 $     48,901            -455    0.94%
2025-12-24   $     48,104 $     48,792            -688    1.43%
2025-12-25   $     48,432 $     48,686            -254    0.52%
2025-12-26   $     48,978 $     48,659            +319    0.65%
2025-12-27   $     49,472 $     48,631            +841    1.70%
2025-12-28   $     49,046 $     48,647            +399    0.81%
2025-12-29   $     48,895 $     48,731            +165    0.34%
2025-12-

In [ ]:
# ─────────────────────────────────────────
# 4. 오차 지표 계산
# ─────────────────────────────────────────
actual = test["close"].values
pred   = test["predicted"].values

mae  = np.mean(np.abs(actual - pred))
rmse = np.sqrt(np.mean((actual - pred) ** 2))
mape = np.mean(np.abs((actual - pred) / actual)) * 100

print("\n[ 오차 지표 ]")
print(f"  MAE  (평균절대오차):  ${mae:,.0f}")
print(f"  RMSE (제곱근평균제곱오차): ${rmse:,.0f}")
print(f"  MAPE (평균절대백분율오차): {mape:.2f}%")



[ 오차 지표 ]
  MAE  (평균절대오차):  $522
  RMSE (제곱근평균제곱오차): $606
  MAPE (평균절대백분율오차): 1.07%


In [ ]:
# ─────────────────────────────────────────
# 5. 내일(2025-12-31) 예측값
# ─────────────────────────────────────────
last_7 = btc["close"].tail(WINDOW)
next_day_pred = last_7.mean()
next_date = btc["date"].max() + pd.Timedelta(days=1)

print(f"\n[ 다음 날 예측 ]")
print(f"  {next_date.date()} 예측 BTC 종가: ${next_day_pred:,.0f}")
print(f"  (직전 7일 종가 평균 기반)")



[ 다음 날 예측 ]
  2025-12-31 예측 BTC 종가: $48,858
  (직전 7일 종가 평균 기반)


In [ ]:
# ─────────────────────────────────────────
# 6. 예측 결과의 한계 및 불확실성 해석
# ─────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════╗
║         예측 결과의 한계 및 불확실성 해석              ║
╚══════════════════════════════════════════════════════╝

【 한계 3가지 】

① 과거 평균 = 미래라는 가정의 취약성
   - MA7은 "최근 7일 가격이 내일을 대표한다"고 가정합니다.
   - 추세가 전환되는 시점(예: 하락→반등)에서 이 가정은
     즉각적으로 무너집니다.
   - 실제로 12월 중순 이후 BTC 급락 구간에서 MA7 예측은
     실제값보다 일관되게 높았습니다(하락 반영 지연).

② 외부 이벤트 반영 불가
   - 규제 발표, 거래소 해킹, ETF 승인 등 돌발 이벤트는
     이동평균 모델이 전혀 감지할 수 없습니다.
   - 이런 이벤트 직후 하루 수천 달러 변동이 발생할 경우,
     MAE/RMSE는 평상시 대비 수배로 폭등합니다.

③ 윈도우(N) 선택에 따른 결과 편차
   - MA7과 MA14는 예측값이 다르며, 정답이 없습니다.
   - 짧은 윈도우 → 노이즈에 민감 / 긴 윈도우 → 반응 지연.
   - 사후에 최적 N을 고르면 과적합(look-ahead bias) 위험이 있습니다.

【 불확실성 / 리스크 2가지 】

① 변동성 급등 리스크
   - 암호화폐는 주식 대비 변동성(Volatility)이 극단적으로 큽니다.
   - MAPE가 2% 이내로 보여도, 가격 수준($50,000 기준)에서
     2%는 이미 $1,000 오차 → 실거래에서 치명적 손실 가능.

② 데이터 구간 선택 편향
   - 이 데이터는 60일치(하락 추세 포함)입니다.
   - 상승장 60일로만 학습하면 정반대 결론이 나올 수 있으며,
     어느 구간을 쓰느냐에 따라 모델 성능 평가가 크게 달라집니다.

【 책임 문장 】

   이 예측 결과는 단순 이동평균(MA7)에 기반한 기준선 모델로,
   실제 투자 의사결정에 사용해서는 안 됩니다.
   모든 예측에는 불확실성이 내재되어 있으며, 예측자(모델)는
   미래 가격을 보장하지 않습니다. 투자 손익에 대한 최종 책임은
   의사결정자 본인에게 있습니다.
""")

print("=" * 55)
print("=" * 55)


╔══════════════════════════════════════════════════════╗
║         예측 결과의 한계 및 불확실성 해석              ║
╚══════════════════════════════════════════════════════╝
 
【 한계 3가지 】
 
① 과거 평균 = 미래라는 가정의 취약성
   - MA7은 "최근 7일 가격이 내일을 대표한다"고 가정합니다.
   - 추세가 전환되는 시점(예: 하락→반등)에서 이 가정은
     즉각적으로 무너집니다.
   - 실제로 12월 중순 이후 BTC 급락 구간에서 MA7 예측은
     실제값보다 일관되게 높았습니다(하락 반영 지연).
 
② 외부 이벤트 반영 불가
   - 규제 발표, 거래소 해킹, ETF 승인 등 돌발 이벤트는
     이동평균 모델이 전혀 감지할 수 없습니다.
   - 이런 이벤트 직후 하루 수천 달러 변동이 발생할 경우,
     MAE/RMSE는 평상시 대비 수배로 폭등합니다.
 
③ 윈도우(N) 선택에 따른 결과 편차
   - MA7과 MA14는 예측값이 다르며, 정답이 없습니다.
   - 짧은 윈도우 → 노이즈에 민감 / 긴 윈도우 → 반응 지연.
   - 사후에 최적 N을 고르면 과적합(look-ahead bias) 위험이 있습니다.
 
【 불확실성 / 리스크 2가지 】
 
① 변동성 급등 리스크
   - 암호화폐는 주식 대비 변동성(Volatility)이 극단적으로 큽니다.
   - MAPE가 2% 이내로 보여도, 가격 수준($50,000 기준)에서
     2%는 이미 $1,000 오차 → 실거래에서 치명적 손실 가능.
 
② 데이터 구간 선택 편향
   - 이 데이터는 60일치(하락 추세 포함)입니다.
   - 상승장 60일로만 학습하면 정반대 결론이 나올 수 있으며,
     어느 구간을 쓰느냐에 따라 모델 성능 평가가 크게 달라집니다.
 
【 책임 문장 】
 
   이 예측 결과는 단순 이동평균(MA7)에 기반한 기준